# Full-Scale TruthfulQA Runner

**817 questions · 3 models (phi3:mini, mistral:7b, llama3:8b) · 4 templates**

### Before running:
1. Runtime → Change runtime type → **A100 GPU** (High-RAM on)
2. Run all cells top to bottom
3. Results are saved to Google Drive so they survive session resets

## Step 1 — Mount Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/llm-hallucination-results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

## Step 2 — Clone repo and install dependencies

In [ ]:
import os
REPO_DIR = '/content/llm-hallucination-phoenix-main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Chief03/llm-hallucination-study {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
    print('Repo already cloned — reset to latest.')

os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
import phoenix, pandas, datasets, openai, scipy, seaborn, matplotlib
print('All dependencies loaded.')

## Step 3 — Install Ollama and start server

In [ ]:
import subprocess, time, requests

!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).ok:
            print('Ollama is ready.')
            break
    except:
        time.sleep(2)

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama --version

## Step 4 — Pull models

In [ ]:
# phi3:mini (~2.2 GB), mistral:7b (~4.1 GB), llama3:8b (~4.7 GB)
# First run takes 10-20 minutes. Subsequent runs use cached models.
for model in ['phi3:mini', 'mistral:7b', 'llama3:8b']:
    print(f'Pulling {model}...')
    !ollama pull {model}
    print(f'{model} ready.')

!ollama list

## Step 5 — Clean rerun (optional)

In [ ]:
# Set to False to resume from a previous checkpoint instead of starting fresh
CLEAN_RERUN = True

if CLEAN_RERUN:
    !rm -f data/experiment_results.csv \
            data/evaluated_results.csv \
            data/metrics.json \
            data/paired_comparisons.json \
            data/plot_*.png
    print('Old outputs removed. Fresh rerun enabled.')
else:
    print('Keeping existing outputs. Checkpoint resume enabled.')

## Step 6 — Run full experiment (checkpoints every 100 rows to Drive)

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# Symlink results file to Drive for persistence
LOCAL_RESULT = 'data/experiment_results.csv'
DRIVE_RESULT = f'{DRIVE_DIR}/experiment_results.csv'

if not os.path.exists(LOCAL_RESULT):
    open(DRIVE_RESULT, 'a').close()
    os.symlink(DRIVE_RESULT, LOCAL_RESULT)
    print(f'Results symlinked to Drive: {DRIVE_RESULT}')
else:
    print('Result file exists — checkpoint resume active.')

!python src/run_experiment.py

## Step 7 — Evaluate metrics

In [ ]:
!python src/evaluate_metrics.py

## Step 8 — Generate plots

In [ ]:
!python src/generate_plots.py

# Copy all outputs to Drive
!cp data/metrics.json {DRIVE_DIR}/
!cp data/evaluated_results.csv {DRIVE_DIR}/
!cp data/paired_comparisons.json {DRIVE_DIR}/
!cp data/plot_*.png {DRIVE_DIR}/
print('All results copied to Google Drive.')

## Step 9 — Validate coverage

In [ ]:
import pandas as pd
df = pd.read_csv('data/experiment_results.csv')
n_items = df['item_id'].nunique()
n_models = df['model'].nunique()
n_templates = df['template'].nunique()
n_prompt_types = df['prompt_type'].nunique()
expected = n_items * n_models * n_templates * n_prompt_types
errors = int(df['output'].astype(str).str.startswith('[ERROR]').sum())

print(f'Rows:      {len(df):,} / {expected:,} expected')
print(f'Items:     {n_items} (should be 817)')
print(f'Models:    {sorted(df["model"].unique().tolist())}')
print(f'Templates: {sorted(df["template"].unique().tolist())}')
print(f'Errors:    {errors} ({errors/len(df)*100:.1f}%)')

## Step 10 — Display plots inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for png in sorted(Path('data').glob('plot_*.png')):
    print(f'--- {png.stem} ---')
    display(Image(str(png)))

## Step 11 — Download all outputs

In [ ]:
!zip -r full_study_outputs.zip data/ reports/
from google.colab import files
files.download('full_study_outputs.zip')

## Optional — Push results to GitHub

Add `GH_TOKEN` to Colab Secrets first (key icon in left sidebar), then run:

```python
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git add -A
!git config user.email 'eworitse2003@gmail.com'
!git config user.name 'Chief03'
!git commit -m 'Add full-study results from Colab' || true
!git remote set-url origin https://{token}@github.com/chief03/llm-hallucination-study.git
!git push origin main
```